In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [2]:
collisions = pd.read_csv(r'../data/raw/dft-road-casualty-statistics-collision-last-5-years.csv, low_memory=False)

                         
print(collisions.columns.tolist())
print(collisions['local_authority_district'].unique()[:30])

SyntaxError: unterminated string literal (detected at line 1) (2897281664.py, line 1)

In [3]:
collisions = pd.read_csv(r'../data/raw/dft-road-casualty-statistics-collision-last-5-years.csv', low_memory=False)

print(collisions.columns.tolist())
print(collisions['local_authority_district'].unique()[:30])

['collision_index', 'collision_year', 'collision_ref_no', 'location_easting_osgr', 'location_northing_osgr', 'longitude', 'latitude', 'police_force', 'collision_severity', 'number_of_vehicles', 'number_of_casualties', 'date', 'day_of_week', 'time', 'local_authority_district', 'local_authority_ons_district', 'local_authority_highway', 'local_authority_highway_current', 'first_road_class', 'first_road_number', 'road_type', 'speed_limit', 'junction_detail_historic', 'junction_detail', 'junction_control', 'second_road_class', 'second_road_number', 'pedestrian_crossing_human_control_historic', 'pedestrian_crossing_physical_facilities_historic', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards_historic', 'carriageway_hazards', 'urban_or_rural_area', 'did_police_officer_attend_scene_of_accident', 'trunk_road_flag', 'lsoa_of_accident_location', 'enhanced_severity_collision', 'collision_injury_based', 

In [4]:
print(collisions['local_authority_ons_district'].unique()[:30])

<StringArray>
['E06000001', 'E06000002', 'E06000004', 'E06000005', 'E06000006', 'E06000007',
 'E06000008', 'E06000009', 'E06000010', 'E06000011', 'E06000012', 'E06000013',
 'E06000014', 'E06000015', 'E06000016', 'E06000017', 'E06000018', 'E06000019',
 'E06000020', 'E06000021', 'E06000022', 'E06000023', 'E06000024', 'E06000025',
 'E06000026', 'E06000027', 'E06000030', 'E06000031', 'E06000032', 'E06000033']
Length: 30, dtype: str


In [5]:
bham = collisions[collisions['local_authority_ons_district'] == 'E08000025']

In [6]:
print('E08000025' in collisions['local_authority_ons_district'].unique())

True


In [7]:
collisions['date'] = pd.to_datetime(collisions['date'], dayfirst=True)
collisions['year'] = collisions['collision_year']  # this file already has collision_year, no need to derive it

bham = collisions[
    (collisions['local_authority_ons_district'] == 'E08000025') &
    (collisions['year'].between(2020, 2024))
].copy()

print(bham.shape)

(10817, 45)


In [8]:
print(bham['year'].value_counts().sort_index())

year
2020    1802
2021    2137
2022    2383
2023    2199
2024    2296
Name: count, dtype: int64


In [9]:
bham_geo = bham.dropna(subset=['longitude', 'latitude'])
print(f"Dropped {len(bham) - len(bham_geo)} rows with missing coordinates")

geometry = [Point(xy) for xy in zip(bham_geo['longitude'], bham_geo['latitude'])]
gdf = gpd.GeoDataFrame(bham_geo, geometry=geometry, crs='EPSG:4326')

# Reproject to British National Grid — needed for accurate distance-based spatial stats later
gdf = gdf.to_crs(epsg=27700)

gdf.to_file(r'../data/processed/bham_crashes_2020_2024.gpkg', driver='GPKG')
bham_geo.to_csv(r'../data/processed/bham_crashes_2020_2024.csv', index=False)

print(f"Saved {len(gdf)} geocoded crashes")

Dropped 0 rows with missing coordinates


C:\ProgramData\anaconda3\envs\crashenv\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()
C:\ProgramData\anaconda3\envs\crashenv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Cannot find tms_NZTM2000.json (GDAL_DATA is not defined)
  ogr_write(


Saved 10817 geocoded crashes


In [10]:
print(gdf.crs)
print(gdf.geometry.iloc[0])

EPSG:27700
POINT (409643.3764208131 290069.6651239754)


In [11]:
bham['year'].value_counts().sort_index()

year
2020    1802
2021    2137
2022    2383
2023    2199
2024    2296
Name: count, dtype: int64

In [12]:
import sqlite3

conn = sqlite3.connect('../sql/crashes.db')
bham_geo.to_sql('collisions', conn, if_exists='replace', index=False)
conn.close()

print("Loaded into SQLite successfully")

Loaded into SQLite successfully


In [13]:
conn = sqlite3.connect('../sql/crashes.db')
check = pd.read_sql("SELECT COUNT(*) as total FROM collisions", conn)
print(check)
conn.close()

   total
0  10817


In [14]:
query = """
SELECT collision_year, collision_severity, COUNT(*) as total
FROM collisions
GROUP BY collision_year, collision_severity
ORDER BY collision_year;
"""

result = pd.read_sql(query, conn)
print(result)

ProgrammingError: Cannot operate on a closed database.

In [15]:
conn = sqlite3.connect('../sql/crashes.db')

query = """
SELECT collision_year, collision_severity, COUNT(*) as total
FROM collisions
GROUP BY collision_year, collision_severity
ORDER BY collision_year;
"""

result = pd.read_sql(query, conn)
print(result)

    collision_year  collision_severity  total
0             2020                   1     10
1             2020                   2    308
2             2020                   3   1484
3             2021                   1     26
4             2021                   2    336
5             2021                   3   1775
6             2022                   1     21
7             2022                   2    397
8             2022                   3   1965
9             2023                   1     19
10            2023                   2    417
11            2023                   3   1763
12            2024                   1     23
13            2024                   2    459
14            2024                   3   1814


In [16]:
conn = sqlite3.connect('../sql/crashes.db')

query = """
SELECT light_conditions, COUNT(*) as total
FROM collisions
GROUP BY light_conditions
ORDER BY total DESC;
"""

result = pd.read_sql(query, conn)
print(result)

   light_conditions  total
0                 1   7188
1                 4   3339
2                 7    160
3                 5     81
4                 6     49


In [17]:
conn = sqlite3.connect('../sql/crashes.db')

query = ""SELECT speed_limit, collision_severity, COUNT(*) as total
FROM collisions
GROUP BY speed_limit, collision_severity
ORDER BY speed_limit; DESC;
"""

result = pd.read_sql(query, conn)
print(result)

SyntaxError: unterminated triple-quoted string literal (detected at line 10) (358663637.py, line 7)

In [18]:
conn = sqlite3.connect('../sql/crashes.db')

query = """
SELECT speed_limit, collision_severity, COUNT(*) as total
FROM collisions
GROUP BY speed_limit, collision_severity
ORDER BY speed_limit;
"""

result = pd.read_sql(query, conn)
print(result)

    speed_limit  collision_severity  total
0            20                   1      7
1            20                   2    130
2            20                   3    504
3            30                   1     72
4            30                   2   1629
5            30                   3   7480
6            40                   1     19
7            40                   2    130
8            40                   3    657
9            50                   2      9
10           50                   3     33
11           60                   2      5
12           60                   3     28
13           70                   1      1
14           70                   2     14
15           70                   3     99


In [19]:
import numpy as np
from shapely.geometry import box
import geopandas as gpd

# Get the bounding box of your crash data (already in EPSG:27700 - meters)
minx, miny, maxx, maxy = gdf.total_bounds

cell_size = 500  # meters - adjust later if needed (smaller = finer detail, needs more data)

cols = np.arange(minx, maxx + cell_size, cell_size)
rows = np.arange(miny, maxy + cell_size, cell_size)

grid_cells = []
for x in cols[:-1]:
    for y in rows[:-1]:
        grid_cells.append(box(x, y, x + cell_size, y + cell_size))

grid = gpd.GeoDataFrame({'geometry': grid_cells}, crs='EPSG:27700')
grid['cell_id'] = range(len(grid))

print(f"Created {len(grid)} grid cells")

Created 1911 grid cells


In [20]:
# Spatial join: assign each crash point to its containing grid cell
joined = gpd.sjoin(gdf, grid, how='left', predicate='within')

# Count crashes per cell
crash_counts = joined.groupby('cell_id').size().reset_index(name='crash_count')

# Merge counts back onto the grid, fill empty cells with 0
grid = grid.merge(crash_counts, on='cell_id', how='left')
grid['crash_count'] = grid['crash_count'].fillna(0).astype(int)

# Drop empty cells with zero neighbors of interest — keep only cells with at least 1 crash 
# OR cells adjacent to crash cells, to avoid a huge empty-ocean-of-zeros skewing results
grid_active = grid[grid['crash_count'] > 0].copy()
print(f"{len(grid_active)} cells contain at least one crash (out of {len(grid)} total)")
print(grid['crash_count'].describe())

961 cells contain at least one crash (out of 1911 total)
count    1911.000000
mean        5.659341
std        10.207974
min         0.000000
25%         0.000000
50%         1.000000
75%         7.000000
max       103.000000
Name: crash_count, dtype: float64


In [21]:
import libpysal as ps
from esda.moran import Moran

# Use Queen contiguity (cells sharing an edge OR corner count as neighbors)
w = ps.weights.contiguity.Queen.from_dataframe(grid, use_index=False)
w.transform = 'r'

y = grid['crash_count'].values

moran = Moran(y, w)
print(f"Moran's I: {moran.I:.4f}")
print(f"Expected I under randomness: {moran.EI:.4f}")
print(f"p-value: {moran.p_sim:.4f}")
print(f"z-score: {moran.z_sim:.4f}")

Moran's I: 0.5634
Expected I under randomness: -0.0005
p-value: 0.0010
z-score: 49.5590


In [22]:
from esda.getisord import G_local

gi_star = G_local(y, w, star=True)

grid['gi_zscore'] = gi_star.Zs
grid['gi_pvalue'] = gi_star.p_sim

# Classify into hot/cold spot confidence tiers (standard convention)
def classify_hotspot(row):
    z, p = row['gi_zscore'], row['gi_pvalue']
    if z > 2.58 and p < 0.01:
        return 'Hot Spot - 99% confidence'
    elif z > 1.96 and p < 0.05:
        return 'Hot Spot - 95% confidence'
    elif z > 1.65 and p < 0.10:
        return 'Hot Spot - 90% confidence'
    elif z < -2.58 and p < 0.01:
        return 'Cold Spot - 99% confidence'
    elif z < -1.96 and p < 0.05:
        return 'Cold Spot - 95% confidence'
    elif z < -1.65 and p < 0.10:
        return 'Cold Spot - 90% confidence'
    else:
        return 'Not Significant'

grid['hotspot_category'] = grid.apply(classify_hotspot, axis=1)

print(grid['hotspot_category'].value_counts())

ImportError: cannot import name 'G_local' from 'esda.getisord' (C:\ProgramData\anaconda3\envs\crashenv\Lib\site-packages\esda\getisord.py)

In [23]:
from esda.getisord import G_Local

gi_star = G_Local(y, w, star=True)

C:\ProgramData\anaconda3\envs\crashenv\Lib\site-packages\esda\getisord.py:421: UserWarning: Gi* requested, but (a) weights are already row-standardized, (b) no weights are on the diagonal, and (c) no default value supplied to star. Assuming that the self-weight is equivalent to the maximum weight in the row. To use a different default (like, .5), set `star=.5`, or use libpysal.weights.fill_diagonal() to set the diagonal values of your weights matrix and use `star=None` in Gi_Local.
  w, star = _infer_star_and_structure_w(w, star, transform)


TypingError: [1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1mFailed in nopython mode pipeline (step: native lowering)
[1m[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1m[1mNo implementation of function Function(<built-in function dot>) found for signature:
 
 >>> dot(array(int64, 2d, C), array(int64, 1d, C), array(int64, 1d, C))
 
There are 4 candidate implementations:
[1m - Of which 2 did not match due to:
 Overload in function 'dot_2': File: numba\np\linalg.py: Line 525.
   With argument(s): '(array(int64, 2d, C), array(int64, 1d, C), array(int64, 1d, C))':[0m
[1m  Rejected as the implementation raised a specific error:
    TypingError: [1mtoo many positional arguments[0m[0m
  raised from C:\ProgramData\anaconda3\envs\crashenv\Lib\site-packages\numba\core\typing\templates.py:791
[1m - Of which 2 did not match due to:
 Overload in function 'dot_3': File: nu

In [24]:
from libpysal.weights import fill_diagonal
from esda.getisord import G_Local

# Rebuild weights with explicit self-weight (standard Gi* approach: cell counts as its own neighbor)
w_star = ps.weights.contiguity.Queen.from_dataframe(grid, use_index=False)
w_star = fill_diagonal(w_star, val=1.0)  # each cell weights itself equally to a neighbor
w_star.transform = 'r'  # row-standardize AFTER filling diagonal

gi_star = G_Local(y, w_star, star=True)

grid['gi_zscore'] = gi_star.Zs
grid['gi_pvalue'] = gi_star.p_sim

TypingError: [1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1mFailed in nopython mode pipeline (step: native lowering)
[1m[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1m[1mNo implementation of function Function(<built-in function dot>) found for signature:
 
 >>> dot(array(int64, 2d, C), array(int64, 1d, C), array(int64, 1d, C))
 
There are 4 candidate implementations:
[1m - Of which 2 did not match due to:
 Overload in function 'dot_2': File: numba\np\linalg.py: Line 525.
   With argument(s): '(array(int64, 2d, C), array(int64, 1d, C), array(int64, 1d, C))':[0m
[1m  Rejected as the implementation raised a specific error:
    TypingError: [1mtoo many positional arguments[0m[0m
  raised from C:\ProgramData\anaconda3\envs\crashenv\Lib\site-packages\numba\core\typing\templates.py:791
[1m - Of which 2 did not match due to:
 Overload in function 'dot_3': File: nu

In [25]:
def classify_hotspot(row):
    z, p = row['gi_zscore'], row['gi_pvalue']
    if z > 2.58 and p < 0.01:
        return 'Hot Spot - 99% confidence'
    elif z > 1.96 and p < 0.05:
        return 'Hot Spot - 95% confidence'
    elif z > 1.65 and p < 0.10:
        return 'Hot Spot - 90% confidence'
    elif z < -2.58 and p < 0.01:
        return 'Cold Spot - 99% confidence'
    elif z < -1.96 and p < 0.05:
        return 'Cold Spot - 95% confidence'
    elif z < -1.65 and p < 0.10:
        return 'Cold Spot - 90% confidence'
    else:
        return 'Not Significant'

grid['hotspot_category'] = grid.apply(classify_hotspot, axis=1)
print(grid['hotspot_category'].value_counts())

KeyError: 'gi_zscore'

In [26]:
y = grid['crash_count'].values.astype(float)

gi_star = G_Local(y, w_star, star=True)

grid['gi_zscore'] = gi_star.Zs
grid['gi_pvalue'] = gi_star.p_sim

C:\ProgramData\anaconda3\envs\crashenv\Lib\site-packages\esda\getisord.py:421: UserWarning: Gi* requested, but (a) weights are already row-standardized, (b) no weights are on the diagonal, and (c) no default value supplied to star. Assuming that the self-weight is equivalent to the maximum weight in the row. To use a different default (like, .5), set `star=.5`, or use libpysal.weights.fill_diagonal() to set the diagonal values of your weights matrix and use `star=None` in Gi_Local.
  w, star = _infer_star_and_structure_w(w, star, transform)


In [27]:
w_star = ps.weights.contiguity.Queen.from_dataframe(grid, use_index=False)
w_star.transform = 'r'

y = grid['crash_count'].values.astype(float)

gi_star = G_Local(y, w_star, star=1.0)  # explicit self-weight = 1.0, no ambiguity, no warning

grid['gi_zscore'] = gi_star.Zs
grid['gi_pvalue'] = gi_star.p_sim

In [28]:
def classify_hotspot(row):
    z, p = row['gi_zscore'], row['gi_pvalue']
    if z > 2.58 and p < 0.01:
        return 'Hot Spot - 99% confidence'
    elif z > 1.96 and p < 0.05:
        return 'Hot Spot - 95% confidence'
    elif z > 1.65 and p < 0.10:
        return 'Hot Spot - 90% confidence'
    elif z < -2.58 and p < 0.01:
        return 'Cold Spot - 99% confidence'
    elif z < -1.96 and p < 0.05:
        return 'Cold Spot - 95% confidence'
    elif z < -1.65 and p < 0.10:
        return 'Cold Spot - 90% confidence'
    else:
        return 'Not Significant'

grid['hotspot_category'] = grid.apply(classify_hotspot, axis=1)
print(grid['hotspot_category'].value_counts())

hotspot_category
Not Significant              1806
Hot Spot - 95% confidence      40
Hot Spot - 99% confidence      36
Hot Spot - 90% confidence      29
Name: count, dtype: int64


In [29]:
grid.to_file(r'../outputs/bham_hotspot_grid.gpkg', driver='GPKG')
print("Saved full grid with hotspot classification")

# Pull out just the 99% confidence hotspots, sorted by strength
top_hotspots = grid[grid['hotspot_category'] == 'Hot Spot - 99% confidence'].sort_values('gi_zscore', ascending=False)

# Get centroid coordinates (in BNG) for each — useful for identifying real-world locations
top_hotspots['centroid'] = top_hotspots.geometry.centroid
top_hotspots_display = top_hotspots[['cell_id', 'crash_count', 'gi_zscore', 'gi_pvalue', 'centroid']].head(15)
print(top_hotspots_display)

Saved full grid with hotspot classification
      cell_id  crash_count  gi_zscore  gi_pvalue  \
1048     1048          103   5.833061      0.001   
1045     1045           98   5.539097      0.001   
948       948           74   4.418356      0.001   
901       901           56   4.253001      0.001   
853       853           59   4.204006      0.001   
904       904           57   3.928414      0.001   
852       852           50   3.812053      0.001   
954       954           60   3.744686      0.001   
903       903           47   3.720189      0.001   
851       851           54   3.585455      0.001   
951       951           50   3.548710      0.001   
902       902           42   3.548710      0.001   
855       855           50   3.481343      0.001   
998       998           40   3.475219      0.001   
900       900           38   3.340485      0.001   

                           centroid  
1048  POINT (408658.536 286042.183)  
1045  POINT (408658.536 284542.183)  
948   POI

In [30]:
top_hotspots_wgs84 = top_hotspots.set_geometry('centroid').set_crs(epsg=27700).to_crs(epsg=4326)
top_hotspots_wgs84['lat'] = top_hotspots_wgs84.geometry.y
top_hotspots_wgs84['lon'] = top_hotspots_wgs84.geometry.x

print(top_hotspots_wgs84[['cell_id', 'crash_count', 'gi_zscore', 'lat', 'lon']].head(15))

      cell_id  crash_count  gi_zscore        lat       lon
1048     1048          103   5.833061  52.472247 -1.873962
1045     1045           98   5.539097  52.458762 -1.874000
948       948           74   4.418356  52.463271 -1.888706
901       901           56   4.253001  52.472268 -1.896044
853       853           59   4.204006  52.476769 -1.903395
904       904           57   3.928414  52.485753 -1.896012
852       852           50   3.812053  52.472274 -1.903405
954       954           60   3.744686  52.490241 -1.888638
903       903           47   3.720189  52.481258 -1.896023
851       851           54   3.585455  52.467779 -1.903414
951       951           50   3.548710  52.476756 -1.888672
902       902           42   3.548710  52.476763 -1.896033
855       855           50   3.481343  52.485759 -1.903375
998       998           40   3.475219  52.467759 -1.881334
900       900           38   3.340485  52.467773 -1.896054


In [31]:
area4 = joined[joined['cell_id'] == 901].copy()
print(f"Crashes in Area 4 cell: {len(area4)}")

# Light conditions breakdown
print("\n--- Light Conditions ---")
print(area4['light_conditions'].value_counts())

# Time of day - extract hour from the time field
area4['hour'] = pd.to_datetime(area4['time'], format='%H:%M', errors='coerce').dt.hour
print("\n--- Crashes by Hour ---")
print(area4['hour'].value_counts().sort_index())

# Quick night vs day split (rough cut: 20:00-05:59 as "night")
area4['night_time'] = area4['hour'].apply(lambda h: 'Night (20:00-05:59)' if pd.notna(h) and (h >= 20 or h < 6) else 'Day (06:00-19:59)')
print("\n--- Night vs Day ---")
print(area4['night_time'].value_counts())

Crashes in Area 4 cell: 56

--- Light Conditions ---
light_conditions
4    30
1    23
5     2
6     1
Name: count, dtype: int64

--- Crashes by Hour ---
hour
0     3
1     3
2     2
3     7
4     2
6     1
7     1
8     1
10    3
11    1
12    2
13    4
14    3
15    4
16    1
17    4
18    3
19    5
20    1
22    2
23    3
Name: count, dtype: int64

--- Night vs Day ---
night_time
Day (06:00-19:59)      33
Night (20:00-05:59)    23
Name: count, dtype: int64


In [32]:
# Export the full grid (with hotspot classification) as CSV with plain lat/lon centroids
grid_export = grid.copy()
grid_export['centroid'] = grid_export.geometry.centroid
grid_export = grid_export.set_geometry('centroid').set_crs(epsg=27700).to_crs(epsg=4326)
grid_export['lat'] = grid_export.geometry.y
grid_export['lon'] = grid_export.geometry.x

grid_export[['cell_id', 'crash_count', 'gi_zscore', 'gi_pvalue', 'hotspot_category', 'lat', 'lon']].to_csv(
    r'../outputs/bham_grid_for_powerbi.csv', index=False
)
print("Exported grid for Power BI")

Exported grid for Power BI


In [33]:
# Export crash-level data WITH the cell_id already attached (from the spatial join you did for Gi*)
joined_export = joined.drop(columns=['geometry', 'index_right'], errors='ignore')
joined_export.to_csv(r'../outputs/bham_crashes_with_cellid.csv', index=False)
print("Exported crash data with cell_id for Power BI")

Exported crash data with cell_id for Power BI


In [34]:
grid_attrs = grid[['cell_id', 'crash_count', 'gi_zscore', 'gi_pvalue', 'hotspot_category']].rename(
    columns={'crash_count': 'cell_total_crash_count'}
)

crashes_flat = joined.merge(grid_attrs, on='cell_id', how='left')
crashes_flat = crashes_flat.drop(columns=['geometry', 'index_right'], errors='ignore')

# Add lat/lon for each individual crash point (not the cell centroid — the actual crash location)
crashes_flat['crash_lat'] = gdf.to_crs(epsg=4326).geometry.y.values
crashes_flat['crash_lon'] = gdf.to_crs(epsg=4326).geometry.x.values

crashes_flat.to_csv(r'../outputs/bham_crashes_flat_for_powerbi.csv', index=False)
print(f"Exported {len(crashes_flat)} rows, single flat table")
print(crashes_flat['hotspot_category'].value_counts())

Exported 10817 rows, single flat table
hotspot_category
Not Significant              7021
Hot Spot - 99% confidence    1760
Hot Spot - 95% confidence    1224
Hot Spot - 90% confidence     810
Name: count, dtype: int64
